# 06 · The QAOA parameter study (67 runs)

**What it does.** Reads the stored results of `domains/vrp/sweep.py`, which varied one factor at a time around
a baseline (p = 2, COBYLA, 300 evaluations, initial γ = 1) on the Stage 1 QUBO (8 qubits) and the route
{1, 2, 5} QUBO (9 qubits): number of layers × optimiser, evaluation budget × optimiser, initial γ, and penalty
weight. It prints the tables and shows the figures.

**Why it was implemented.** The brief (§5, "Quantum algorithm & tuning") asks to investigate QAOA depth,
parameter initialisation and the optimiser, and to document the parameter choices. The study is also what
selected the reported configuration (p = 4, Powell).

**Resources.**
- Code: `domains/vrp/sweep.py` (plan, runs, figures), `configs/vrp.yaml` (`sweep` block).
- Data: `results/sweep/summary.csv` (one row per run), `results/sweep/stage1|stage2/*.json` (full distributions), `results/sweep/fig_*.png`.
- Libraries: SciPy optimisers COBYLA (Powell 1994), Nelder-Mead (Nelder & Mead 1965), Powell (Powell 1964).

Re-running the study takes about 4 minutes: `python -m domains.vrp.sweep --config configs/vrp.yaml`.

**Metrics.** `p_optimal` = share of shots on an optimal bitstring; `p_feasible` = share breaking no constraint;
`lift` = p_optimal divided by that of uniform sampling (1/256 for Stage 1, 2/512 for the route).

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
import csv

rows = list(csv.DictReader(Path("results/sweep/summary.csv").open(encoding="utf-8")))
print("runs:", len(rows))

def table(qubo, experiment, columns=("reps", "optimizer", "maxiter", "initial_gamma", "penalty")):
    sel = [r for r in rows if r["qubo"] == qubo and experiment in r["experiments"].split("+")]
    print(f"\n{qubo} | {experiment}")
    print("".join(f"{c:>14}" for c in columns) + f"{'p_optimal':>11}{'p_feasible':>12}{'lift':>7}{'evals':>7}{'CNOTs':>7}")
    for r in sorted(sel, key=lambda r: [float(r[c]) if c != "optimizer" else 0 for c in columns] + [r["optimizer"]]):
        print("".join(f"{r[c]:>14}" for c in columns) + f"{float(r['p_optimal']):>11.3f}{float(r['p_feasible']):>12.3f}"
              f"{float(r['lift_vs_uniform']):>7.1f}{r['n_evaluations']:>7}{r['hw_cx']:>7}")

for qubo in ("stage1", "stage2"):
    table(qubo, "reps", ("reps", "optimizer"))

In [ ]:
import matplotlib.pyplot as plt

def show(path, size=(12, 8)):
    fig, ax = plt.subplots(figsize=size)
    ax.imshow(plt.imread(path))
    ax.axis("off")
    plt.show()

show("results/sweep/fig_reps.png")

**Reading:** depth helps the route QUBO up to p = 3–4; at p = 5 (10 angles) the 300-evaluation budget runs out.
Stage 1 shows no clear depth trend. Nelder-Mead degrades to uniform sampling at p ≥ 4.

## Evaluation budget
The x-axis of the figure is the number of evaluations actually used: COBYLA and Powell stop on their own.

In [ ]:
for qubo in ("stage1", "stage2"):
    table(qubo, "maxiter", ("maxiter", "optimizer"))
show("results/sweep/fig_budget.png")

## Initial γ and penalty weight

In [ ]:
for qubo in ("stage1", "stage2"):
    table(qubo, "initial_gamma", ("initial_gamma",))
    table(qubo, "penalty", ("penalty",))
show("results/sweep/fig_gamma.png")
show("results/sweep/fig_penalty.png")

## Full measurement distributions of the best runs

In [ ]:
show("results/sweep/fig_distribution_stage1.png", (12, 4.5))
show("results/sweep/fig_distribution_stage2.png", (12, 4.5))
best = {q: max((r for r in rows if r["qubo"] == q), key=lambda r: float(r["p_optimal"])) for q in ("stage1", "stage2")}
for q, r in best.items():
    print(f"best {q}: p = {r['reps']}, {r['optimizer']}, P = {r['penalty']}: p_optimal {float(r['p_optimal']):.3f}, "
          f"p_feasible {float(r['p_feasible']):.3f}")

## Expected results
- 67 runs.
- Best Stage 1 run: p = 4, Powell, p_optimal **0.052** (lift 13). Best route run: p = 4, COBYLA, p_optimal **0.185** (lift 47); p = 4 Powell gives 0.177.
- Initial γ moves route p_optimal from 0.029 (γ = 0.4) to 0.145 (γ = 2): a 5× spread from the starting point alone.
- Penalty: flat for the route (0.074–0.079), not monotone for Stage 1 (0.002–0.028).
- Each setting used one starting point, so single values are noisy; read trends, not decimals.

In [ ]:
def find(qubo, **kw):
    return [r for r in rows if r["qubo"] == qubo and all(r[k] == str(v) for k, v in kw.items())]

assert len(rows) == 67
assert best["stage1"]["reps"] == "4" and best["stage1"]["optimizer"] == "Powell"
assert abs(float(best["stage2"]["p_optimal"]) - 0.185) < 0.002
gammas = {r["initial_gamma"]: float(r["p_optimal"]) for r in rows
          if r["qubo"] == "stage2" and "initial_gamma" in r["experiments"].split("+")}
assert abs(gammas["2.0"] / gammas["0.4"] - 5) < 0.1